# 09 — TP4: Serialización del modelo de producción

Paso 1 del TP4: dejar listo el **artefacto de producción** que consumirá la app
Streamlit. Tres entregables en `app/models/`:

1. `clasificador_riesgo.joblib` — el `Pipeline` **completo** (imputación +
   escalado + one-hot + `XGBClassifier`) en un solo objeto serializado.
2. `metadata.json` — rangos de las predictoras, categorías, umbral de decisión,
   métricas de referencia, importancias, envolvente para validación de inputs y
   la distribución de `n_focos` para el gráfico de contexto.

El modelo es el **mejor del TP3** (`XGBClassifier`, con los hiperparámetros ya
tuneados) reentrenado sobre el **dataset completo**.

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (average_precision_score, precision_score,
                             recall_score, f1_score)

RAIZ = Path.cwd()
proj = RAIZ if (RAIZ / "src").exists() else RAIZ.parent
sys.path.insert(0, str(proj))
from src.data import cargar_dataset, preparar_features, UMBRAL_RIESGO
from src import modelos as M

RSTATE = 42
MODELS_DIR = proj / "app" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Artefactos ->", MODELS_DIR)

Artefactos -> C:\Users\juanc\OneDrive\Desktop\PatagonIA\PatagonIA\app\models


## 1. Datos completos

Cargamos las **1981 filas** (ya sin el cuadrante de flaring). A diferencia del
TP3, **no** hacemos `train_test_split`: el modelo de producción se entrena con
**todos** los datos disponibles.

In [2]:
df = cargar_dataset(str(proj / "data" / "processed" / "patagonia_ia.csv"))
X, y_cont, y = preparar_features(df)      # X: 8 predictoras, y: riesgo_alto

# Verificación anti-fuga (igual que en TP3): ninguna variable de fuego en X.
COLS_FUEGO = ["n_focos", "brillo_medio", "brillo_max", "frp_medio", "frp_max",
              "brillo_t31_medio", "pct_noche", "pct_verano", "pct_conf_alta",
              "n_anios_activo", "mes_pico"]
assert not [c for c in COLS_FUEGO if c in X.columns], "FUGA en X"
assert list(X.columns) == M.NUMERICAS + [M.CATEGORICA]

print("X:", X.shape, "| positivos:", int(y.sum()), f"({100*y.mean():.2f} %)")
SPW = (y == 0).sum() / (y == 1).sum()       # scale_pos_weight sobre TODO el set
print(f"scale_pos_weight (full) = {SPW:.3f}")

X: (1981, 8) | positivos: 271 (13.68 %)
scale_pos_weight (full) = 6.310


## 2. Por qué entrenar sobre el dataset completo

La **evaluación** del modelo (comparar familias, elegir hiperparámetros, medir
generalización) ya se hizo en el TP3 con **validación cruzada** y un test
retenido: ahí es imprescindible separar train/test para no auto-engañarse.

Pero una vez **decidido** el modelo, el artefacto que va a producción debe
aprovechar **toda** la evidencia disponible: reservar un 20 % para test en
producción sería tirar datos útiles sin ganar nada (ya no estamos midiendo).
Por eso el modelo final se entrena sobre las 1981 filas. Las métricas que
reportamos como referencia siguen siendo las **honestas del TP3** (CV /
multi-semilla), no las de resustitución sobre estos mismos datos.

## 3. Entrenamiento del Pipeline final

Mismos hiperparámetros tuneados del TP3 para `XGBClassifier`
(`n_estimators=300`, `max_depth=4`, `learning_rate=0.05`, `subsample=0.9`,
`colsample_bytree=0.9`, `eval_metric='aucpr'`), con `scale_pos_weight`
recalculado sobre el dataset completo. Todo el preprocesamiento viaja **dentro**
del mismo `Pipeline` (`src.modelos.construir_pipeline_clf`): un único objeto que
recibe un DataFrame crudo y devuelve la predicción.

In [3]:
xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, eval_metric="aucpr",
    scale_pos_weight=SPW, random_state=RSTATE, n_jobs=-1)

pipeline = M.construir_pipeline_clf(xgb)
pipeline.fit(X, y)
print("Pipeline entrenado sobre", X.shape[0], "filas.")
print(pipeline)

Pipeline entrenado sobre 1981 filas.
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['elevacion', 'temp_media',
                                                   'precip_anual',
                                                   'viento_medio',
                                                   'humedad_relativa',
                                                   'dist_asentamiento_km',
                                                   'dist_ruta_km']),
                                                 ('cat',
                                                  Pi

## 4. Métricas de referencia (honestas, del TP3)

El umbral de decisión de producción es **0.386**, el que en el TP3 fijaba
**recall ≈ 0.80** (priorizar no perder hexágonos de riesgo sobre la precision).
Para reportar `recall`/`precision`/`AUC-PR` de forma **honesta** (no
resustitución) usamos `cross_val_predict` con probabilidades *out-of-fold* sobre
el mismo `StratifiedKFold(5)` del TP3, y evaluamos en ese umbral. El F1 headline
(0.60 ± 0.07) proviene del experimento multi-semilla del TP3.

In [4]:
UMBRAL = 0.386
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSTATE)

proba_oof = cross_val_predict(
    M.construir_pipeline_clf(
        XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                      subsample=0.9, colsample_bytree=0.9, eval_metric="aucpr",
                      scale_pos_weight=SPW, random_state=RSTATE, n_jobs=-1)),
    X, y, cv=skf, method="predict_proba")[:, 1]

pred_oof = (proba_oof >= UMBRAL).astype(int)
metricas = {
    "f1_pos_media": 0.60,          # headline multi-semilla (TP3)
    "f1_pos_desvio": 0.07,
    "auc_pr": float(average_precision_score(y, proba_oof)),
    "umbral_decision": UMBRAL,
    "recall_en_umbral": float(recall_score(y, pred_oof, zero_division=0)),
    "precision_en_umbral": float(precision_score(y, pred_oof, zero_division=0)),
    "f1_en_umbral": float(f1_score(y, pred_oof, zero_division=0)),
}
for k, v in metricas.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

  f1_pos_media: 0.6000
  f1_pos_desvio: 0.0700
  auc_pr: 0.6481
  umbral_decision: 0.3860
  recall_en_umbral: 0.7196
  precision_en_umbral: 0.4588
  f1_en_umbral: 0.5603


## 5. Metadata para la app

`metadata.json` desacopla la app del interior del modelo. Incluye:

- **rangos** (min/max/mediana + unidad) de cada predictora numérica → acota los
  sliders y define su valor por defecto.
- **categorías** de `cobertura_veg` → opciones del `selectbox`.
- **umbral** de decisión (0.386) y **umbral_riesgo_focos** (150).
- **métricas** de referencia (sección 4).
- **importancias** agregadas a las 8 features originales (las componentes
  one-hot de `cobertura_veg` se suman).
- **envolvente**: media y covarianza de las 7 numéricas + un `cutoff` de
  distancia de Mahalanobis, para avisar si una combinación ingresada cae fuera
  de la nube observada (p. ej. precipitación alta con humedad baja, que no
  coexisten).
- **distribucion_n_focos**: los valores de `n_focos` para el histograma de
  contexto.

In [5]:
ETIQUETAS = {
    "elevacion": "Elevación (m)",
    "temp_media": "Temperatura media anual (°C)",
    "precip_anual": "Precipitación anual (mm)",
    "viento_medio": "Viento medio (m/s)",
    "humedad_relativa": "Humedad relativa (%)",
    "dist_asentamiento_km": "Distancia a asentamiento (km)",
    "dist_ruta_km": "Distancia a ruta (km)",
}
UNIDADES = {"elevacion": "m", "temp_media": "°C", "precip_anual": "mm",
            "viento_medio": "m/s", "humedad_relativa": "%",
            "dist_asentamiento_km": "km", "dist_ruta_km": "km"}

# --- Rangos (imputamos el único nulo con la mediana para las estadísticas) ---
Xnum = X[M.NUMERICAS].copy()
Xnum = Xnum.fillna(Xnum.median())
rangos = {}
for c in M.NUMERICAS:
    rangos[c] = {
        "min": float(Xnum[c].min()), "max": float(Xnum[c].max()),
        "mediana": float(Xnum[c].median()), "unidad": UNIDADES[c],
        "etiqueta": ETIQUETAS[c],
    }

# --- Categorías de cobertura_veg (ordenadas por frecuencia) ---
categorias = X[M.CATEGORICA].value_counts().index.tolist()

# --- Importancias agregadas a las 8 features originales ---
prep = pipeline.named_steps["prep"]
nombres = list(prep.get_feature_names_out())
imp_raw = pipeline.named_steps["model"].feature_importances_
importancias = {}
for c in M.NUMERICAS:
    importancias[c] = float(imp_raw[nombres.index(f"num__{c}")])
importancias[M.CATEGORICA] = float(
    sum(v for n, v in zip(nombres, imp_raw) if n.startswith("cat__")))

# --- Envolvente: media + covarianza + cutoff de Mahalanobis (7 numéricas) ---
mu = Xnum.mean().values
cov = np.cov(Xnum.values, rowvar=False)
inv = np.linalg.pinv(cov)
d = np.sqrt(np.einsum("ij,jk,ik->i", Xnum.values - mu, inv, Xnum.values - mu))
cutoff = float(np.percentile(d, 97.5))     # 97.5 % de los hexágonos por debajo
print(f"Mahalanobis: mediana={np.median(d):.2f} | cutoff(97.5%)={cutoff:.2f}")

envolvente = {
    "features": M.NUMERICAS,
    "media": mu.tolist(),
    "cov": cov.tolist(),
    "cutoff_mahalanobis": cutoff,
}

Mahalanobis: mediana=2.36 | cutoff(97.5%)=4.39


In [6]:
metadata = {
    "modelo": "XGBClassifier",
    "descripcion": ("Clasificador de riesgo alto de incendio "
                    "(riesgo_alto = n_focos > 150) a partir de 8 predictoras "
                    "ambientales. Pipeline: imputación + escalado + one-hot + "
                    "XGBoost."),
    "entrenado_sobre": "dataset completo (1981 hexágonos H3, sin flaring)",
    "n_muestras": int(X.shape[0]),
    "prevalencia": float(y.mean()),
    "scale_pos_weight": float(SPW),
    "features_numericas": M.NUMERICAS,
    "feature_categorica": M.CATEGORICA,
    "etiquetas": ETIQUETAS,
    "rangos": rangos,
    "categorias_cobertura": categorias,
    "umbral_decision": UMBRAL,
    "umbral_riesgo_focos": int(UMBRAL_RIESGO),
    "metricas": metricas,
    "importancias": importancias,
    "envolvente": envolvente,
    "distribucion_n_focos": [int(v) for v in y_cont.tolist()],
}

ruta_meta = MODELS_DIR / "metadata.json"
with open(ruta_meta, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("metadata.json ->", ruta_meta, f"({ruta_meta.stat().st_size/1024:.1f} KB)")
print("claves:", list(metadata))

metadata.json -> C:\Users\juanc\OneDrive\Desktop\PatagonIA\PatagonIA\app\models\metadata.json (21.8 KB)
claves: ['modelo', 'descripcion', 'entrenado_sobre', 'n_muestras', 'prevalencia', 'scale_pos_weight', 'features_numericas', 'feature_categorica', 'etiquetas', 'rangos', 'categorias_cobertura', 'umbral_decision', 'umbral_riesgo_focos', 'metricas', 'importancias', 'envolvente', 'distribucion_n_focos']


## 6. Serialización del Pipeline y prueba de recarga

Guardamos con `joblib` y verificamos de inmediato que el objeto recargado
predice **idéntico** al original sobre una muestra (garantiza que la app usará
exactamente el mismo modelo).

In [7]:
ruta_modelo = MODELS_DIR / "clasificador_riesgo.joblib"
joblib.dump(pipeline, ruta_modelo)
print("modelo ->", ruta_modelo, f"({ruta_modelo.stat().st_size/1024:.1f} KB)")

# --- Prueba de recarga ---
cargado = joblib.load(ruta_modelo)
muestra = X.sample(20, random_state=RSTATE)
p_orig = pipeline.predict_proba(muestra)[:, 1]
p_load = cargado.predict_proba(muestra)[:, 1]
assert np.allclose(p_orig, p_load), "El modelo recargado difiere del original"
print("OK — el modelo recargado reproduce las probabilidades del original.")

# Demostración end-to-end: un hexágono nuevo (dict crudo -> predicción).
ejemplo = pd.DataFrame([{
    "elevacion": 350.0, "temp_media": 9.0, "precip_anual": 1100.0,
    "viento_medio": 3.0, "humedad_relativa": 82.0,
    "dist_asentamiento_km": 110.0, "dist_ruta_km": 120.0,
    "cobertura_veg": "bosque",
}])
prob = float(cargado.predict_proba(ejemplo)[:, 1][0])
print(f"\nEjemplo (bosque húmedo remoto): prob_riesgo={prob:.3f} -> "
      f"{'RIESGO ALTO' if prob >= UMBRAL else 'RIESGO BAJO'} (umbral {UMBRAL})")

modelo -> C:\Users\juanc\OneDrive\Desktop\PatagonIA\PatagonIA\app\models\clasificador_riesgo.joblib (436.4 KB)
OK — el modelo recargado reproduce las probabilidades del original.

Ejemplo (bosque húmedo remoto): prob_riesgo=0.464 -> RIESGO ALTO (umbral 0.386)


**Listo.** `app/models/` contiene el modelo y la metadata. El Paso 2 (la app
Streamlit) sólo carga estos dos archivos: no re-entrena ni necesita el dataset
crudo en tiempo de ejecución.